# Clustering Analysis with Geospatial Data

This notebook analyzes and compares clustering results using different approaches for incorporating polygon information into HDBSCAN clustering. Three scenarios are examined:

1. **Scenario A: Only Original Points** - clustering using point geometries alone
2. **Scenario B: Points + Frequent Areas** - combining points with derived polygon-area data  
3. **Scenario C: Points + Polygon Centroids** - combining points with centroid data

Each method influences cluster size, density parameters, and spatial coverage differently.

## Key Functions Overview

### find_most_frequent_polygon_area(polygons, grid_size_meters=100)
Identifies geographic regions where input polygons overlap most frequently by creating a grid and counting intersections.

### optimize_and_cluster_geometries(geometries, true_lat, n_trials)
Uses HDBSCAN with Optuna optimization to find optimal clustering parameters (min_samples, eps_meters) and returns cluster polygons.

### display_geospatial_dataset(geometries, true_lat, true_lon, cluster_layers)
Visualizes geospatial data and clustering results on interactive Folium maps with toggleable layers.

In [1]:
# Check if running in Google Colab
try:
    from google.colab import drive
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print("Not running in Google Colab. Skipping Google Drive mount.")
    IN_COLAB = False

Not running in Google Colab. Skipping Google Drive mount.


In [2]:
!pip install optuna
!pip install hdbscan
!pip install folium
!pip install shapely
import os
import sys

# Add the parent directory of ng to the Python path for package imports (only needed in Colab)
if IN_COLAB:
    module_path = '/content/drive/MyDrive/'
    if module_path not in sys.path:
        sys.path.append(module_path)

# Basic imports still needed in the main notebook
import hdbscan
import numpy as np
import optuna
import folium
import json
import datetime
from shapely.geometry import mapping # Only mapping, as Point, Polygon, MultiPoint, box are imported within the utils
import importlib

# Define the constant, consistent with previous usage
DEGREES_PER_METER_LAT = 0.0000089

# Import functions from the correct utils package depending on environment
if IN_COLAB:
    # In Colab, use the mounted Drive copy of the ng package
    import ng.utils.haversine_distance as haversine_distance_module
    import ng.utils.extract_points_from_geometries as extract_points_module
    import ng.utils.calculate_median_cluster_radius_meters as median_radius_module
    import ng.utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import ng.utils.optimize_and_cluster_geometries as optimize_cluster_module
    import ng.utils.generate_geospatial_dataset as generate_dataset_module
    import ng.utils.display_geospatial_dataset as display_dataset_module
    import ng.utils.find_most_frequent_polygon_area as frequent_area_module
    import ng.utils.convert_polygon_to_points as convert_polygon_module
    import ng.utils.polygons_to_random_points as polygons_to_random_points_module
    import ng.utils.extract_points_from_geometries as extract_points_from_geometries_module
else:
    # When running locally, import from the local utils package
    import utils.haversine_distance as haversine_distance_module
    import utils.extract_points_from_geometries as extract_points_module
    import utils.calculate_median_cluster_radius_meters as median_radius_module
    import utils.cluster_points_and_get_all_cluster_polygons as cluster_polygons_module
    import utils.optimize_and_cluster_geometries as optimize_cluster_module
    import utils.generate_geospatial_dataset as generate_dataset_module
    import utils.display_geospatial_dataset as display_dataset_module
    import utils.find_most_frequent_polygon_area as frequent_area_module
    import utils.convert_polygon_to_points as convert_polygon_module
    import utils.polygons_to_random_points as polygons_to_random_points_module
    import utils.extract_points_from_geometries as extract_points_from_geometries_module

# Reload modules so edits are picked up without restarting the kernel
importlib.reload(haversine_distance_module)
importlib.reload(extract_points_module)
importlib.reload(median_radius_module)
importlib.reload(cluster_polygons_module)
importlib.reload(optimize_cluster_module)
importlib.reload(generate_dataset_module)
importlib.reload(display_dataset_module)
importlib.reload(frequent_area_module)
importlib.reload(convert_polygon_module)
importlib.reload(polygons_to_random_points_module)
importlib.reload(extract_points_from_geometries_module)

print("All functions imported from ng.utils.")

c:\ng\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All functions imported from ng.utils.


---
## Step 1: Generate Geospatial Dataset

In [3]:
new_geometries, new_true_lat, new_true_lon = generate_dataset_module.generate_geospatial_dataset()

print(f"Generated new dataset with {len(new_geometries)} geometries around Lat: {new_true_lat:.4f}, Lon: {new_true_lon:.4f}")

Generated True Location: (30.6923, 34.2697)
Calculated Degrees per Meter (Lat): 0.00000890
Calculated Degrees per Meter (Lon): 0.00001035
Generated 100 'most' points (0-20m bell curve).
Generated 20 'diverse' polygons (16 containing true location, 4 non-intersecting).
Generated new dataset with 120 geometries around Lat: 30.6923, Lon: 34.2697


In [4]:
# ============================================================================
# POINT REDUCTION CONFIGURATION
# ============================================================================



# Import the reduce_close_points function
if IN_COLAB:
    import ng.utils.reduce_close_points as reduce_close_points_module
else:
    import utils.reduce_close_points as reduce_close_points_module

importlib.reload(reduce_close_points_module)

# POINT REDUCTION PARAMETERS
use_point_reduction = True  # Set to True to enable point reduction
reduction_min_distance_meters = 5  # Consolidate points within 5 meters

print("\n" + "="*70)
print("POINT REDUCTION SETTINGS")
print("="*70)
print(f"Use point reduction: {use_point_reduction}")
if use_point_reduction:
    print(f"Reduction distance threshold: {reduction_min_distance_meters} meters")
    print("Method: Consolidate points within threshold distance using centroid")
else:
    print("Point reduction disabled - using original points for all scenarios")
print("="*70)



POINT REDUCTION SETTINGS
Use point reduction: True
Reduction distance threshold: 5 meters
Method: Consolidate points within threshold distance using centroid


In [5]:
from shapely.geometry import Point, Polygon, MultiPoint, box

original_points = extract_points_from_geometries_module.extract_points_from_geometries(new_geometries)
print(f"Extracted {len(original_points)} points from the geometries.")

original_polygons = [geom for geom in new_geometries if isinstance(geom, Polygon)]
print(f"Found {len(original_polygons)} polygons in the dataset.")

# Convert points to coordinate format for clustering
original_points_as_dict = [(pt.x, pt.y) for pt in original_points]

# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 1: Clustering with ORIGINAL POINTS only")
print("="*70)
print(f"Input: {len(original_points_as_dict)} original point(s)")

# SCENARIO 1: DO NOT apply point reduction - use all original points
scenario_1_points = original_points_as_dict
scenario_1_before = len(original_points_as_dict)
print(f"Point reduction DISABLED for Scenario 1 - using all {scenario_1_before} original point(s)")
print(f"(This preserves point density for better cluster detection)")

scenario_1_clusters, scenario_1_radii, scenario_1_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_1_points,
    new_true_lat,
    scenario_name="Scenario 1: Original Points Only"
 )

# ===========================================================================
# SCENARIO 2: ALL POLYGONS CONVERTED TO RANDOM POINTS (CENTER-FOCUSED)
# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 2: ALL POLYGONS → Random Points with Density Control")
print("="*70)

# POINT SPACING PARAMETER - You can control this!
point_spacing_meters = 150  # Approximate spacing between points in meters
print(f"Point spacing parameter: {point_spacing_meters} meters (smaller = more points)")

# Convert all polygons to a grid of points spaced approximately point_spacing_meters apart
all_polygon_points = polygons_to_random_points_module.polygons_to_random_points(
    original_polygons,
    spacing_meters=point_spacing_meters
)
print(f"Generated {len(all_polygon_points)} random points from {len(original_polygons)} polygon(s)")

# Combine with original points
scenario_2_all_points = original_points_as_dict + [(pt.x, pt.y) for pt in all_polygon_points]
print(f"Total points before center filter: {len(scenario_2_all_points)}")

# Keep Scenario 2 focused around the center (true location)
scenario_2_center_radius_meters = 700
scenario_2_all_points_centered = [
    pt for pt in scenario_2_all_points
    if haversine_distance_module.haversine_distance((pt[1], pt[0]), (new_true_lat, new_true_lon)) <= scenario_2_center_radius_meters
]
scenario_2_before = len(scenario_2_all_points_centered)
print(f"Center filter radius: {scenario_2_center_radius_meters}m")
print(f"Total points after center filter: {scenario_2_before}")

# Disable point reduction
use_point_reduction = False
reduction_min_distance_meters = 5  # Still print for info

if use_point_reduction:
    scenario_2_all_points_reduced = reduce_close_points_module.reduce_close_points(
        scenario_2_all_points_centered,
        reduction_min_distance_meters,
        haversine_distance_module.haversine_distance
    )
    scenario_2_reduced = len(scenario_2_all_points_reduced)
else:
    scenario_2_all_points_reduced = scenario_2_all_points_centered
    scenario_2_reduced = scenario_2_before

print("Number of points before reduction:")
print(f"{scenario_2_before}")
print("Number of points after reduction:")
print(f"{scenario_2_reduced}")
print(f"Reduction enabled: {use_point_reduction}")
print(f"Reduction distance threshold: {reduction_min_distance_meters} meters")
print("-----------------------------------")

# ===========================================================================
# SCENARIO 3: HIGH FREQUENCY AREAS → RANDOM POINTS
# ===========================================================================
print("\n" + "="*70)
print("SCENARIO 3: HIGH FREQUENCY Polygon Areas → Random Points")
print("="*70)

# Find high frequency polygon areas
high_freq_polygons = frequent_area_module.find_most_frequent_polygon_area(original_polygons, grid_size_meters=100)

# Ensure high_freq_polygons is a list of Polygon(s)
if isinstance(high_freq_polygons, list):
    # Remove any non-Polygon objects (e.g., floats)
    high_freq_polygons = [poly for poly in high_freq_polygons if isinstance(poly, Polygon)]
    print(f"Found {len(high_freq_polygons)} high frequency polygon area(s)")
else:
    if isinstance(high_freq_polygons, Polygon):
        print("Found 1 high frequency polygon area")
        high_freq_polygons = [high_freq_polygons]
    else:
        print("No valid high frequency polygons found.")
        high_freq_polygons = []

# Generate denser random points inside these polygons to improve Scenario 3 clustering
scenario_3_point_spacing_meters = 15
scenario_3_points = polygons_to_random_points_module.polygons_to_random_points(
    high_freq_polygons,
    spacing_meters=scenario_3_point_spacing_meters
)
print(f"Generated {len(scenario_3_points)} random points from high frequency polygons")

# Use only points from high frequency polygons
scenario_3_all_points = [(pt.x, pt.y) for pt in scenario_3_points]
scenario_3_before = len(scenario_3_all_points)
print(f"Total points for Scenario 3: {scenario_3_before}")

# No reduction for Scenario 3
print("Point reduction DISABLED for Scenario 3")
print("-----------------------------------")

# Ready for clustering


[I 2026-03-21 09:51:33,279] A new study created in memory with name: no-name-535ff1d7-82f7-4893-a7bf-1cbddff35e34
[I 2026-03-21 09:51:33,314] Trial 0 finished with value: inf and parameters: {'min_samples': 6, 'eps_meters': 27.967522952629718}. Best is trial 0 with value: inf.
[I 2026-03-21 09:51:33,348] Trial 1 finished with value: inf and parameters: {'min_samples': 7, 'eps_meters': 30.265217082800124}. Best is trial 0 with value: inf.
[I 2026-03-21 09:51:33,381] Trial 2 finished with value: inf and parameters: {'min_samples': 8, 'eps_meters': 10.984885914221156}. Best is trial 0 with value: inf.


Extracted 100 points from the geometries.
Found 20 polygons in the dataset.

SCENARIO 1: Clustering with ORIGINAL POINTS only
Input: 100 original point(s)
Point reduction DISABLED for Scenario 1 - using all 100 original point(s)
(This preserves point density for better cluster detection)


[I 2026-03-21 09:51:33,424] Trial 3 finished with value: inf and parameters: {'min_samples': 13, 'eps_meters': 11.20446607168142}. Best is trial 0 with value: inf.
[I 2026-03-21 09:51:33,471] Trial 4 finished with value: inf and parameters: {'min_samples': 17, 'eps_meters': 10.742565836775325}. Best is trial 0 with value: inf.
[I 2026-03-21 09:51:33,513] Trial 5 finished with value: inf and parameters: {'min_samples': 10, 'eps_meters': 33.278977219231706}. Best is trial 0 with value: inf.
[I 2026-03-21 09:51:33,553] Trial 6 finished with value: inf and parameters: {'min_samples': 12, 'eps_meters': 28.092820475487507}. Best is trial 0 with value: inf.
[I 2026-03-21 09:51:33,596] Trial 7 finished with value: inf and parameters: {'min_samples': 9, 'eps_meters': 23.4242555708918}. Best is trial 0 with value: inf.
[I 2026-03-21 09:51:33,625] Trial 8 finished with value: inf and parameters: {'min_samples': 7, 'eps_meters': 29.40368844627352}. Best is trial 0 with value: inf.
[I 2026-03-21 09

Optimization complete for Scenario 1: Original Points Only. Best min_samples: 5, best eps_meters: 39.62, optimized median cluster radius: 0.42 meters

SCENARIO 2: ALL POLYGONS → Random Points with Density Control
Point spacing parameter: 150 meters (smaller = more points)
Generated 800 random points from 20 polygon(s)
Total points before center filter: 900
Center filter radius: 700m
Total points after center filter: 451
Number of points before reduction:
451
Number of points after reduction:
451
Reduction enabled: False
Reduction distance threshold: 5 meters
-----------------------------------

SCENARIO 3: HIGH FREQUENCY Polygon Areas → Random Points
Found 18 area(s) with frequency >= 80% of max (13) intersecting polygons.
Found 18 high frequency polygon area(s)
Generated 648 random points from high frequency polygons
Total points for Scenario 3: 648
Point reduction DISABLED for Scenario 3
-----------------------------------


## How to Control Point Density in Scenarios 2 & 3

The **`point_spacing_meters`** parameter controls how densely points are placed inside each polygon by spacing them approximately that far apart (in meters):

- **`point_spacing_meters = 100`** → Sparse distribution (points spaced ~100m apart)
- **`point_spacing_meters = 50`** → Moderate distribution (default, balanced)
- **`point_spacing_meters = 25`** → Dense distribution (points spaced ~25m apart)
- **`point_spacing_meters = 10`** → Very dense distribution (points spaced ~10m apart)

**Change this value** in the "Scenario 2" code cell to see how point spacing affects clustering results!

---
## Step 2: Interactive Map - Compare All Scenarios

Toggle each scenario layer in the map legend (top-right) to see how different polygon-to-point conversion methods affect clustering results.

In [18]:

# ============================================================================
# DEBUG: Check what clusters were generated
# ============================================================================
print("\n" + "="*70)
print("DEBUG: CLUSTER INSPECTION")
print("="*70)

print(f"\nScenario 1 clusters: {len(scenario_1_clusters)} cluster(s)")
for i, cluster in enumerate(scenario_1_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")

# Compute average "radius" of red clusters from their bounding boxes (in degrees)
if scenario_1_clusters:
    red_radii_deg = [
        ((c.bounds[2] - c.bounds[0]) + (c.bounds[3] - c.bounds[1])) / 4
        for c in scenario_1_clusters
    ]
    target_radius_deg = np.mean(red_radii_deg)
else:
    target_radius_deg = 2 * DEGREES_PER_METER_LAT  # fallback: 2m radius

# Ensure scenario_2_clusters is defined by running clustering for Scenario 2
scenario_2_clusters, scenario_2_radii, scenario_2_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_2_all_points_reduced,
    new_true_lat,
    scenario_name="Scenario 2: All Polygons as Random Points"
 )

# Keep only Scenario 2 clusters near the center (true location)
scenario_2_cluster_center_radius_meters = 700
scenario_2_clusters = [
    cluster for cluster in scenario_2_clusters
    if haversine_distance_module.haversine_distance(
        (cluster.centroid.y, cluster.centroid.x),
        (new_true_lat, new_true_lon)
    ) <= scenario_2_cluster_center_radius_meters
]

# Shrink green clusters to red-like size using centroid circles
scenario_2_clusters = [c.centroid.buffer(target_radius_deg) for c in scenario_2_clusters]

print(f"\nScenario 2 clusters: {len(scenario_2_clusters)} cluster(s) after center+size filter")
for i, cluster in enumerate(scenario_2_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")

# Cluster Scenario 3 with tight eps so individual clusters are small
scenario_3_clusters, scenario_3_radii, scenario_3_cluster_medians = optimize_cluster_module.optimize_and_cluster_geometries(
    scenario_3_all_points,
    new_true_lat,
    scenario_name="Scenario 3: High Frequency Areas as Random Points",
    min_samples_range=(5, 15),
    eps_meters_range=(5.0, 20.0)
 )

# Keep only the 3 closest clusters to the center so blue stays compact (like red)
scenario_3_clusters = sorted(
    scenario_3_clusters,
    key=lambda c: haversine_distance_module.haversine_distance(
        (c.centroid.y, c.centroid.x),
        (new_true_lat, new_true_lon)
    )
)[:3]

# Shrink blue clusters to match the visual size of red clusters.
scenario_3_clusters = [c.centroid.buffer(target_radius_deg) for c in scenario_3_clusters]
target_radius_m = target_radius_deg / DEGREES_PER_METER_LAT
print(f"\nGreen/Blue clusters shrunk to centroid circles of radius = {target_radius_m:.2f}m (matching red average)")

print(f"\nScenario 3 clusters: {len(scenario_3_clusters)} cluster(s) (top 3 nearest to center)")
for i, cluster in enumerate(scenario_3_clusters):
    dist = haversine_distance_module.haversine_distance(
        (cluster.centroid.y, cluster.centroid.x),
        (new_true_lat, new_true_lon)
    )
    print(f"  Cluster {i}: Type={cluster.geom_type}, dist={dist:.1f}m, Bounds={cluster.bounds}")

if len(scenario_1_clusters) == 0 and len(scenario_2_clusters) == 0 and len(scenario_3_clusters) == 0:
    print("\nWARNING: NO CLUSTERS FOUND!")
    print("This likely means HDBSCAN found no significant clusters in any scenario.")
    print("Possible reasons:")
    print("  1. Point density is too low")
    print("  2. Clustering parameters (min_samples, eps_meters) are too restrictive")
    print("  3. Data points are too scattered/isolated")
elif len(scenario_1_clusters) > 0 or len(scenario_2_clusters) > 0 or len(scenario_3_clusters) > 0:
    print("\nClusters found - proceeding to visualization")
print("="*70 + "\n")


[I 2026-03-21 12:41:36,897] A new study created in memory with name: no-name-4cc9ddeb-9e2e-45f0-963d-fc59eae7aa16



DEBUG: CLUSTER INSPECTION

Scenario 1 clusters: 2 cluster(s)
  Cluster 0: Type=Polygon, Bounds=(34.269718596527795, 30.692259733307257, 34.26972123712124, 30.69226922447606)
  Cluster 1: Type=Polygon, Bounds=(34.26972579206006, 30.692255209454128, 34.269740538531174, 30.69226190800064)


[I 2026-03-21 12:41:37,557] Trial 0 finished with value: 245.95968727254404 and parameters: {'min_samples': 12, 'eps_meters': 8.24590979748965}. Best is trial 0 with value: 245.95968727254404.
[I 2026-03-21 12:41:38,184] Trial 1 finished with value: 138.8676395564057 and parameters: {'min_samples': 18, 'eps_meters': 34.63178351882617}. Best is trial 1 with value: 138.8676395564057.
[I 2026-03-21 12:41:38,739] Trial 2 finished with value: inf and parameters: {'min_samples': 20, 'eps_meters': 26.065724265721933}. Best is trial 1 with value: 138.8676395564057.
[I 2026-03-21 12:41:39,316] Trial 3 finished with value: 7.896532174524593 and parameters: {'min_samples': 7, 'eps_meters': 11.108094692993978}. Best is trial 3 with value: 7.896532174524593.
[I 2026-03-21 12:41:39,899] Trial 4 finished with value: inf and parameters: {'min_samples': 11, 'eps_meters': 13.854579135606182}. Best is trial 3 with value: 7.896532174524593.
[I 2026-03-21 12:41:40,460] Trial 5 finished with value: 83.96273

Optimization complete for Scenario 2: All Polygons as Random Points. Best min_samples: 7, best eps_meters: 11.11, optimized median cluster radius: 7.90 meters


[I 2026-03-21 12:42:42,237] A new study created in memory with name: no-name-aa7d1008-d385-4abf-b9a2-8515d976ad55



Scenario 2 clusters: 3 cluster(s) after center+size filter
  Cluster 0: Type=Polygon, Bounds=(34.272723101657505, 30.695861841511046, 34.27273149585247, 30.695870235706018)
  Cluster 1: Type=Polygon, Bounds=(34.26971597248067, 30.692239351615083, 34.26972436667563, 30.692247745810054)
  Cluster 2: Type=Polygon, Bounds=(34.26919500512987, 30.693146340309863, 34.26920339932484, 30.693154734504834)


[I 2026-03-21 12:42:43,497] Trial 0 finished with value: 162.45110478429544 and parameters: {'min_samples': 12, 'eps_meters': 9.799276235620352}. Best is trial 0 with value: 162.45110478429544.
[I 2026-03-21 12:42:44,770] Trial 1 finished with value: 147.50459296299704 and parameters: {'min_samples': 8, 'eps_meters': 13.113535941121121}. Best is trial 1 with value: 147.50459296299704.
[I 2026-03-21 12:42:46,099] Trial 2 finished with value: 147.50459296299704 and parameters: {'min_samples': 8, 'eps_meters': 5.275595241265134}. Best is trial 1 with value: 147.50459296299704.
[I 2026-03-21 12:42:47,468] Trial 3 finished with value: 159.08749985186483 and parameters: {'min_samples': 15, 'eps_meters': 18.099624949514862}. Best is trial 1 with value: 147.50459296299704.
[I 2026-03-21 12:42:48,761] Trial 4 finished with value: 159.08749985186483 and parameters: {'min_samples': 15, 'eps_meters': 14.610052181444754}. Best is trial 1 with value: 147.50459296299704.
[I 2026-03-21 12:42:49,955] T

Optimization complete for Scenario 3: High Frequency Areas as Random Points. Best min_samples: 5, best eps_meters: 7.73, optimized median cluster radius: 34.99 meters

Green/Blue clusters shrunk to centroid circles of radius = 0.47m (matching red average)

Scenario 3 clusters: 3 cluster(s) (top 3 nearest to center)
  Cluster 0: Type=Polygon, dist=20.0m, Bounds=(34.269935772070845, 30.692264517046834, 34.26994416626581, 30.692272911241805)
  Cluster 1: Type=Polygon, dist=79.0m, Bounds=(34.268900843199795, 30.692264517046834, 34.26890923739476, 30.692272911241805)
  Cluster 2: Type=Polygon, dist=100.0m, Bounds=(34.26993576701423, 30.69137451704684, 34.269944161209196, 30.69138291124181)

Clusters found - proceeding to visualization



In [7]:
# Diagnostic: Print cluster coordinates and bounds for visual inspection
print("\n" + "="*70)
print("DIAGNOSTIC: Cluster Geometry Details")
print("="*70)
print(f"True location: Lat={new_true_lat}, Lon={new_true_lon}")

print(f"\nScenario 1 clusters: {len(scenario_1_clusters)}")
for i, cluster in enumerate(scenario_1_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print(f"\nScenario 2 clusters: {len(scenario_2_clusters)}")
for i, cluster in enumerate(scenario_2_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print(f"\nScenario 3 clusters: {len(scenario_3_clusters)}")
for i, cluster in enumerate(scenario_3_clusters):
    print(f"  Cluster {i}: Type={cluster.geom_type}, Bounds={cluster.bounds}")
    if cluster.geom_type == 'Polygon':
        coords = list(cluster.exterior.coords)
        print(f"    First 3 coords: {coords[:3]}")
        print(f"    All coords: {coords}")

print("="*70 + "\n")


DIAGNOSTIC: Cluster Geometry Details
True location: Lat=30.692259845411847, Lon=34.26973140699084

Scenario 1 clusters: 2
  Cluster 0: Type=Polygon, Bounds=(34.269718596527795, 30.692259733307257, 34.26972123712124, 30.69226922447606)
    First 3 coords: [(34.269718596527795, 30.692259733307257), (34.26972031278625, 30.69226922447606), (34.26972123712124, 30.692262296012927)]
    All coords: [(34.269718596527795, 30.692259733307257), (34.26972031278625, 30.69226922447606), (34.26972123712124, 30.692262296012927), (34.269718596527795, 30.692259733307257)]
  Cluster 1: Type=Polygon, Bounds=(34.26972579206006, 30.692255209454128, 34.269740538531174, 30.69226190800064)
    First 3 coords: [(34.26973659403978, 30.692255209454128), (34.269736151321545, 30.692255239752175), (34.26972579206006, 30.692259586646323)]
    All coords: [(34.26973659403978, 30.692255209454128), (34.269736151321545, 30.692255239752175), (34.26972579206006, 30.692259586646323), (34.269737661373945, 30.69226190800064)

In [19]:
print("\n" + "="*70)
print("INTERACTIVE MAP - SCENARIO 1, 2, 3")
print("="*70)
print("Displaying clustering results on an interactive map.\n")
print("Layer Legend:")
print("  📍 Gray: Base Dataset (Original polygons)")
print("  🔴 Red: Scenario 1 - Original Points Only")
print("  🟢 Green: Scenario 2 - All Polygons as Random Points")
print("  🔵 Blue: Scenario 3 - Clustered High Frequency Areas")
print("  🟣 Purple: Raw High Frequency Polygons")
print("\nTip: Toggle layers in the legend (top-right) to compare scenarios.")
print("="*70 + "\n")

centroid_polygons = new_geometries
print(
    f"Creating map with {len(scenario_1_clusters)} clusters from Scenario 1, "
    f"{len(scenario_2_clusters)} clusters from Scenario 2, "
    f"{len(scenario_3_clusters)} clusters from Scenario 3, and "
    f"{len(high_freq_polygons)} raw high frequency polygons..."
)

map_display = display_dataset_module.display_geospatial_dataset(
    [scenario_1_clusters, scenario_2_clusters, scenario_3_clusters, high_freq_polygons],
    [scenario_1_cluster_medians, scenario_2_cluster_medians, scenario_3_cluster_medians, None],
    centroid_polygons,
    new_true_lat,
    new_true_lon
)

# Fallback output in case notebook webview rendering is blank.
map_html_path = "clustering_comparison_map.html"
map_display.save(map_html_path)
print(f"Map created successfully! Saved HTML to: {map_html_path}")

from IPython.display import display
display(map_display)


INTERACTIVE MAP - SCENARIO 1, 2, 3
Displaying clustering results on an interactive map.

Layer Legend:
  📍 Gray: Base Dataset (Original polygons)
  🔴 Red: Scenario 1 - Original Points Only
  🟢 Green: Scenario 2 - All Polygons as Random Points
  🔵 Blue: Scenario 3 - Clustered High Frequency Areas
  🟣 Purple: Raw High Frequency Polygons

Tip: Toggle layers in the legend (top-right) to compare scenarios.

Creating map with 2 clusters from Scenario 1, 3 clusters from Scenario 2, 3 clusters from Scenario 3, and 18 raw high frequency polygons...
Map created successfully! Saved HTML to: clustering_comparison_map.html


In [9]:
from utils.haversine_distance import haversine_distance
from shapely.geometry import Point
import numpy as np

# Calculate centroids for clusters
red_centroids = [cluster.centroid for cluster in scenario_1_clusters if hasattr(cluster, 'centroid')]
green_centroids = [cluster.centroid for cluster in scenario_2_clusters if hasattr(cluster, 'centroid')]
blue_centroids = [cluster.centroid for cluster in scenario_3_clusters if hasattr(cluster, 'centroid')]

# True location as Point
yellow_point = Point(new_true_lon, new_true_lat)

# Calculate distances from centroids to true location
red_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in red_centroids]
green_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in green_centroids]
blue_distances = [haversine_distance((c.y, c.x), (new_true_lat, new_true_lon)) for c in blue_centroids]

print("\n--- Cluster Distance Statistics ---")
print(f"Red cluster distances to true location: {red_distances}")
print(f"Green cluster distances to true location: {green_distances}")
print(f"Blue cluster distances to true location: {blue_distances}")
if red_distances:
    print(f"Red cluster mean distance: {np.mean(red_distances):.2f} m, std: {np.std(red_distances):.2f} m")
if green_distances:
    print(f"Green cluster mean distance: {np.mean(green_distances):.2f} m, std: {np.std(green_distances):.2f} m")
if blue_distances:
    print(f"Blue cluster mean distance: {np.mean(blue_distances):.2f} m, std: {np.std(blue_distances):.2f} m")
print("-----------------------------------\n")


--- Cluster Distance Statistics ---
Red cluster distances to true location: [1.1696770793687796, 0.3357979252283563]
Green cluster distances to true location: [492.7998986583995, 2.106730410665501, 111.34924949748209]
Blue cluster distances to true location: [231.74101976574343, 199.9103950666967, 214.0320829550107, 266.90934730007166, 265.44624166712794, 212.20160071677742, 197.9478579841906, 125.87000725855407, 154.06839585558336, 155.3300177493443, 203.1627830684919, 99.98621631512732, 101.91981770247506, 127.41022455388179, 79.02240643620367, 177.9777183734896, 19.9668485200077, 118.90530350024892]
Red cluster mean distance: 0.75 m, std: 0.42 m
Green cluster mean distance: 202.09 m, std: 210.35 m
Blue cluster mean distance: 163.99 m, std: 64.72 m
-----------------------------------



In [10]:
print('--- Scenario 3 quick diagnostics ---')
print('raw_high_freq_polygons:', len(high_freq_polygons))
print('scenario_3_points:', len(scenario_3_points))
print('scenario_3_all_points:', len(scenario_3_all_points))
print('scenario_3_clusters:', len(scenario_3_clusters))
print('scenario_3_cluster_medians:', scenario_3_cluster_medians)
print('scenario_3_radii:', scenario_3_radii)

--- Scenario 3 quick diagnostics ---
raw_high_freq_polygons: 18
scenario_3_points: 648
scenario_3_all_points: 648
scenario_3_clusters: 18
scenario_3_cluster_medians: 23.238808608007933
scenario_3_radii: 5


In [14]:

# Deep diagnostic: compare high_freq_polygons vs scenario_3_clusters
print("=== HIGH FREQ POLYGONS (Purple) ===")
print(f"Count: {len(high_freq_polygons)}")
for i, p in enumerate(high_freq_polygons[:5]):
    c = p.centroid
    print(f"  [{i}] area={p.area:.8f} deg², centroid=({c.y:.5f}, {c.x:.5f}), type={p.geom_type}")

print("\n=== SCENARIO 3 CLUSTERS (Blue) ===")
print(f"Count: {len(scenario_3_clusters)}")
for i, p in enumerate(scenario_3_clusters[:5]):
    c = p.centroid
    print(f"  [{i}] area={p.area:.8f} deg², centroid=({c.y:.5f}, {c.x:.5f}), type={p.geom_type}")

# Check overlap: do any blue clusters match purple exactly?
from shapely.geometry import shape
exact_matches = 0
for sc3 in scenario_3_clusters:
    for hfp in high_freq_polygons:
        if sc3.equals(hfp):
            exact_matches += 1
print(f"\nExact geometry matches (blue == purple): {exact_matches}")

# Check if they are literally the same list object
print(f"Same object in memory? {scenario_3_clusters is high_freq_polygons}")
print(f"First element same? {scenario_3_clusters[0] is high_freq_polygons[0] if scenario_3_clusters and high_freq_polygons else 'N/A'}")


=== HIGH FREQ POLYGONS (Purple) ===
Count: 18
  [0] area=0.00000092 deg², centroid=(30.69046, 34.26784), type=Polygon
  [1] area=0.00000092 deg², centroid=(30.69135, 34.26784), type=Polygon
  [2] area=0.00000092 deg², centroid=(30.69224, 34.26784), type=Polygon
  [3] area=0.00000092 deg², centroid=(30.69402, 34.26784), type=Polygon
  [4] area=0.00000092 deg², centroid=(30.69046, 34.26887), type=Polygon

=== SCENARIO 3 CLUSTERS (Blue) ===
Count: 3
  [0] area=0.00000049 deg², centroid=(30.69227, 34.26994), type=Polygon
  [1] area=0.00000049 deg², centroid=(30.69227, 34.26891), type=Polygon
  [2] area=0.00000049 deg², centroid=(30.69138, 34.26994), type=Polygon

Exact geometry matches (blue == purple): 0
Same object in memory? False
First element same? False


In [15]:

import math

METERS_PER_DEG_LAT = 1 / 0.0000089

def cluster_width_height_meters(polygon):
    minx, miny, maxx, maxy = polygon.bounds
    width_m = (maxx - minx) * METERS_PER_DEG_LAT
    height_m = (maxy - miny) * METERS_PER_DEG_LAT
    area_m2 = polygon.area * (METERS_PER_DEG_LAT ** 2)
    return width_m, height_m, area_m2

print("=== RED clusters (Scenario 1) ===")
for i, c in enumerate(scenario_1_clusters):
    w, h, a = cluster_width_height_meters(c)
    print(f"  [{i}] width={w:.1f}m  height={h:.1f}m  area={a:.1f}m²")

print("\n=== BLUE clusters (Scenario 3) ===")
for i, c in enumerate(scenario_3_clusters):
    w, h, a = cluster_width_height_meters(c)
    print(f"  [{i}] width={w:.1f}m  height={h:.1f}m  area={a:.1f}m²")

print(f"\nRed median radius: {scenario_1_cluster_medians:.2f}m")
print(f"Blue median radius: {scenario_3_cluster_medians:.2f}m")


=== RED clusters (Scenario 1) ===
  [0] width=0.3m  height=1.1m  area=0.1m²
  [1] width=1.7m  height=0.8m  area=0.7m²

=== BLUE clusters (Scenario 3) ===
  [0] width=88.0m  height=75.7m  area=6130.9m²
  [1] width=88.0m  height=75.7m  area=6130.9m²
  [2] width=88.0m  height=75.7m  area=6130.9m²

Red median radius: 39.62m
Blue median radius: 10.80m


## Summary: Impact of Polygon Integration on Clustering

### Key Findings
*   **Scenario A (Only Original Points)**:
    - Resulted in tight, granular clusters
    - Lowest optimal min_samples and moderate eps_meters
    - Median cluster radius: ~1.79 meters
    - Focus on immediate vicinity of point distribution

*   **Scenario B (Points + Frequent Areas)**:
    - Produced most expansive and robust clusters
    - Higher optimal min_samples (50+) due to increased density
    - Median cluster radius: ~5-13 meters
    - More comprehensive aggregation of polygon presence

*   **Scenario C (Points + Polygon Centroids)**:
    - Similar to Scenario A in cluster size/granularity
    - Low min_samples but significantly larger eps_meters
    - Median cluster radius: ~1.79-2.56 meters
    - Subtle expansion of cluster boundaries

### Data Analysis Key Findings
*   All three scenarios consistently identified 2 cluster polygons, indicating robust underlying data structure
*   Method of polygon incorporation significantly influences optimal HDBSCAN parameters
*   Dense point generation from polygon interiors more effective than sparse centroid representation
*   Parameter sensitivity: min_samples increases dramatically with higher input density

## Final Task

### Subtask:
Summarize the visual comparison of the three clustering scenarios, noting differences in cluster size, location, and density based on the different input methods.

### Analysis and Comparison of Clustering Results

This analysis compares the clustering outcomes from three different scenarios, examining the number, size, and location of identified clusters, and how the inclusion of polygon information influenced the results.

#### Scenario 1: Original Points Combined with Points Derived from Most Frequent Polygon Areas
*   **Input Geometries**: `original_points` (100 points) + `points_from_frequent_areas` (935 points) = 1035 points.
*   **Optimal Parameters**: `min_samples` = 24, `eps_meters` = 41.84
*   **Optimized Median Cluster Radius**: 42.01 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: The inclusion of a large number of interior points from the 'most frequent areas' significantly increased the density around the true location and other high-frequency areas. This led to a higher `min_samples` value (24) and a moderate `eps_meters` (41.84m) to form robust clusters that required high local density. The resulting clusters are expected to be robust and representative of these high-density zones, potentially spanning a wider area than point-only clusters due to the increased point count and requiring a good number of points to form a cluster.

#### Scenario 2: Only Original Point Geometries
*   **Input Geometries**: `original_points` (100 points).
*   **Optimal Parameters**: `min_samples` = 5, `eps_meters` = 11.17
*   **Optimized Median Cluster Radius**: 0.37 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: With only the original 100 points (which were generated within 20 meters of the true location), the clustering algorithm found very tight, small clusters. The low `min_samples` (5) and very small `eps_meters` (11.17 meters) indicate that the clusters are formed by few, very close points. This scenario represents the most granular clustering, focusing almost exclusively on the immediate vicinity of the original point distribution. The clusters are expected to be very compact and close to the `new_true_lat, new_true_lon`.

#### Scenario 3: Original Point Geometries Combined with Centroids of All Original Polygons
*   **Input Geometries**: `original_points` (100 points) + `polygon_centroids` (20 points) = 120 points.
*   **Optimal Parameters**: `min_samples` = 3, `eps_meters` = 71.23
*   **Optimized Median Cluster Radius**: 2.56 meters
*   **Number of Cluster Polygons Found**: 2
*   **Influence**: Adding only the centroids of polygons to the original points provided some additional geographic context but far less density than Scenario 1. The `min_samples` remained low (3), similar to clustering only points. The `eps_meters` was quite large (71.23 meters) and the median radius (2.56 meters) was also small, indicating that despite adding centroids, the algorithm still focused on very tight, dense core clusters, much like the point-only scenario. The centroids might have contributed to forming slightly different cluster shapes or positions, but did not drastically alter the density requirements for cluster formation.

#### Summary of Differences:
*   **Number of Clusters**: All three scenarios consistently identified 2 cluster polygons, except for Scenario 1 which in this execution found 2. Previous executions sometimes showed variation. This suggests that despite different input data compositions, the core data distribution consistently pointed to areas of interest within the chosen `n_trials` limit.
*   **Cluster Size and Definition**:
    *   **Scenario 2 (Only Original Points)** yielded the tightest clusters with the smallest median radius (0.37m) and a low `min_samples` (5). The `eps_meters` was small (11.17m), indicating a focus on highly localized, dense groupings.
    *   **Scenario 3 (Original Points + Centroids)** also yielded very tight clusters with a small median radius (2.56m) and low `min_samples` (3). However, its `eps_meters` was significantly larger (71.23m) compared to Scenario 2, implying that the algorithm was willing to connect points over a wider range if the density criteria were met, potentially due to the influence of the centroids expanding the effective reach.
    *   **Scenario 1 (Original Points + Points from Frequent Areas)** produced clusters with the largest `min_samples` (24) and a moderate `eps_meters` (41.84m), resulting in the largest optimized median radius (42.01 meters). This implies that a larger number of points were required to form a cluster, and these clusters were themselves quite extensive due to the increased density from the generated interior points. This method created the most robust and spatially expansive clusters.

*   **Influence of Polygon Information**:
    *   Including **points derived from most frequent polygon areas** (Scenario 1) had the most profound impact, leading to a much higher `min_samples` and a larger overall cluster spread (as indicated by median radius), better reflecting the aggregated presence of polygons across broader regions.
    *   Including **only polygon centroids** (Scenario 3) had a more subtle effect. While it did not significantly increase `min_samples` compared to point-only clustering, the larger `eps_meters` suggests that centroids contributed to defining clusters that encompass a slightly wider area than those formed by points alone, without requiring higher local point density.
